# 04 · RAG evaluation (Modules 9 & 10)

Retrieval, generation and engineering metrics for each strategy.

Run `hcrag make-evalset` and `hcrag compare` first, or execute the cells below.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
import json
from pathlib import Path

from src.evaluation.dataset import load_evaluation_set

questions = load_evaluation_set()
print(f"{len(questions)} questions")

qframe = pd.DataFrame([q.model_dump() for q in questions])
print(qframe["category"].value_counts())
print(f"\nlabelled (usable for retrieval metrics): {qframe['relevant_document_ids'].str.len().gt(0).sum()}")
print(f"unanswerable (refusal probes):           {qframe['unanswerable'].sum()}")
qframe[["question", "category", "unanswerable"]].head(8)

## Strategy comparison

In [ ]:
path = Path("../data/evaluation/results/strategy_comparison.json")
if path.exists():
    payload = json.loads(path.read_text())
else:
    from src.evaluation.experiments import run_strategy_comparison
    payload = run_strategy_comparison().to_dict()

rows = []
for name, report in payload["arms"].items():
    rows.append({
        "strategy": name,
        **{k: report["retrieval_document"].get(k) for k in
           ["recall@5", "recall@20", "precision@5", "mrr", "ndcg@5"]},
        "faithfulness": report["generation"]["faithfulness"],
        "answer_relevance": report["generation"]["answer_relevance"],
        "citation_accuracy": report["generation"]["citation_accuracy"],
        "latency_s": report["engineering"]["mean_total_latency"],
    })

comparison = pd.DataFrame(rows)
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
comparison.plot.bar(x="strategy", y=["recall@5", "recall@20", "mrr"], ax=axes[0],
                    title="Retrieval quality", rot=20)
comparison.plot.bar(x="strategy", y=["faithfulness", "answer_relevance", "citation_accuracy"],
                    ax=axes[1], title="Generation quality", rot=20)
for ax in axes:
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
ax = comparison.plot.scatter(x="latency_s", y="recall@5", s=120, color="#4C9BE8",
                             title="Quality vs latency")
for _, row in comparison.iterrows():
    ax.annotate(row["strategy"], (row["latency_s"], row["recall@5"]),
                textcoords="offset points", xytext=(6, 6), fontsize=8)
plt.tight_layout()

## Hallucination behaviour\n\nThe most important table in the project: does the system refuse when it should?

In [ ]:
safety = pd.DataFrame([{
    "strategy": name,
    "refusal_on_unanswerable": report["generation"]["refusal_rate_unanswerable"],
    "false_refusal": report["generation"]["false_refusal_rate"],
    "citation_accuracy": report["generation"]["citation_accuracy"],
    "citation_coverage": report["generation"]["citation_coverage"],
} for name, report in payload["arms"].items()])
safety.round(3)

`refusal_on_unanswerable` should be near 1.0 (refuses fabricated questions) and `false_refusal` near 0.0 (still answers real ones). Those two together are the hallucination-control summary.

## Per-question inspection\n\nAverages hide failures. Look at the worst cases.

In [ ]:
runs = sorted(Path("../data/evaluation/results").glob("*_2*.json"),
              key=lambda p: p.stat().st_mtime, reverse=True)
if runs:
    detail = pd.DataFrame(json.loads(runs[0].read_text())["per_question"])
    if "faithfulness" in detail:
        worst = detail.nsmallest(5, "faithfulness")[
            ["question", "faithfulness", "n_citations", "refused"]]
        display(worst)
    detail["latency"].plot.hist(bins=30, color="#5BB98C", title="Per-question latency (s)")
    plt.tight_layout()

## Is the benchmark measuring retrieval, or string overlap?

The first comparison run reported BM25 as the best retriever with a **perfect Recall@5**. A perfect score is a warning sign, not a result — so before believing it, measure how much of each question's vocabulary was simply copied from the passage it was generated from.

In [ ]:
from src.evaluation.dataset import load_evaluation_set, stratify_by_leakage
from src.retrieval.index import RagIndex

index = RagIndex.load("default")
questions = load_evaluation_set()

leak = pd.DataFrame([{
    "question": q.question,
    "overlap": q.lexical_overlap,
    "category": q.category,
} for q in questions if q.relevant_chunk_ids])

print(leak["overlap"].describe().round(3))
leak["overlap"].plot.hist(bins=20, color="#C0504D",
                          title="Fraction of question terms copied from the gold passage")
plt.axvline(0.5, ls="--", c="k", label="stratification threshold")
plt.legend(); plt.tight_layout()

In [ ]:
# The worst offenders are effectively restatements of their own answer.
leak.nlargest(5, "overlap")[["overlap", "question"]]

### Stratified comparison

Filtering leaky questions away would be the wrong fix: some overlap is unavoidable (a question about amlodipine has to say "amlodipine"). Instead, score the slices separately.

In [ ]:
from src.evaluation.experiments import DEFAULT_ARMS
from src.evaluation.rag_evaluation import run_evaluation

strata = stratify_by_leakage(questions, threshold=0.5)
print({k: len(v) for k, v in strata.items()})

rows = []
for slice_name in ["low_leakage", "high_leakage"]:
    for name, cfg in DEFAULT_ARMS:
        m = run_evaluation(config=cfg, questions=strata[slice_name],
                           use_llm=False, index=index, save=False).retrieval_doc
        rows.append({"slice": slice_name, "strategy": name,
                     "recall@1": m.recall.get(1, 0), "mrr": m.mrr, "ndcg@5": m.ndcg.get(5, 0)})

stratified = pd.DataFrame(rows)
stratified.pivot(index="strategy", columns="slice", values=["recall@1", "mrr"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, slice_name in zip(axes, ["high_leakage", "low_leakage"]):
    sub = stratified[stratified["slice"] == slice_name].set_index("strategy")
    sub[["recall@1", "mrr"]].plot.bar(ax=ax, title=slice_name, rot=20)
    ax.set_ylim(0.75, 1.0)
plt.tight_layout()

**The conclusion.** On leaky questions BM25 beats dense retrieval by ~9 points of Recall@1. On clean questions that advantage disappears completely — BM25 and vector score identically — and hybrid + reranking wins, which is what the architecture predicted.

BM25's apparent superiority was an artifact of how the test set was built. This is why the arm chosen for expensive generation metrics is ranked on the low-leakage slice, not the aggregate.

## Caveat\n\nSynthetic labels mark only the source chunk as relevant, so a retriever that finds an equally correct passage elsewhere is scored wrong. Absolute recall is therefore **under-reported**; the bias is identical across strategies, so the comparison holds.